# 🏠 House Prices — Data Cleaning

**Goal:** Handle missing values, remove outliers, and save clean datasets for EDA & Feature Engineering.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


## 1. Load Data

In [2]:
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)


Train shape: (1460, 81)
Test shape : (1459, 80)


## 2. Save Test IDs & Drop Id Column

`Id` is just an index — not a feature. Save test IDs separately for submission later.

In [3]:
# Save test IDs for final submission
test_ids = test["Id"].copy()
test_ids.to_csv("../data/test_ids.csv", index=False)

# Drop Id from both
train = train.drop("Id", axis=1)
test  = test.drop("Id", axis=1)

print("Id column dropped.")


Id column dropped.


## 3. Remove Outliers

Kaggle discussions identify 2 well-known outliers: large GrLivArea but very low SalePrice (data entry errors).

In [4]:
before = len(train)

train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000))]

after = len(train)
print(f"Outliers removed: {before - after} rows")
print(f"Train size after outlier removal: {after}")


Outliers removed: 2 rows
Train size after outlier removal: 1458


## 4. Explore Missing Values

In [5]:
missing_pct = (train.isnull().sum() / len(train) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

missing_df = pd.DataFrame({
    "Missing Count": train.isnull().sum()[missing_pct.index],
    "Missing %"    : missing_pct
})
missing_df


,Missing Count,Missing %
PoolQC,1452,99.588477
MiscFeature,1404,96.296296
Alley,1367,93.758573
Fence,1177,80.727023
MasVnrType,872,59.807956
FireplaceQu,690,47.325103
LotFrontage,259,17.764060
GarageType,81,5.555556
GarageYrBlt,81,5.555556
GarageFinish,81,5.555556


In [6]:
print("Numerical columns :", train.select_dtypes(include=["int64","float64"]).shape[1])
print("Categorical columns:", train.select_dtypes(include=["object"]).shape[1])


Numerical columns : 37
Categorical columns: 43


## 5. Make Clean Copies

Always work on copies — never modify the raw dataframes directly.

In [7]:
train_df = train.copy()
test_df  = test.copy()


## 6. Categorical — Fill 'None'

These columns are missing because the house simply doesn't have that feature (e.g. no pool, no alley). NaN means 'None' here.

In [8]:
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond",
    "MasVnrType"
]

for col in none_cols:
    train_df[col] = train_df[col].fillna("None")
    test_df[col]  = test_df[col].fillna("None")

print("None-fill done for", len(none_cols), "columns")


None-fill done for 15 columns


## 7. LotFrontage — Group Median by Neighborhood

Lot frontage varies a lot by neighborhood. Fill missing with the median of each neighborhood group.

In [9]:
for df in [train_df, test_df]:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"] \
                          .transform(lambda x: x.fillna(x.median()))

print("LotFrontage nulls remaining:", train_df["LotFrontage"].isnull().sum())


LotFrontage nulls remaining: 0


## 8. Garage Numeric Columns — Fill 0

Houses without garages → 0 is correct, not median.

In [10]:
garage_num_cols = ["GarageYrBlt", "GarageCars", "GarageArea"]

for col in garage_num_cols:
    train_df[col] = train_df[col].fillna(0)
    test_df[col]  = test_df[col].fillna(0)

train_df["MasVnrArea"] = train_df["MasVnrArea"].fillna(0)
test_df["MasVnrArea"]  = test_df["MasVnrArea"].fillna(0)

print("Garage & MasVnrArea zeros filled.")


Garage & MasVnrArea zeros filled.


## 9. Basement Numeric — Fill 0

In [11]:
bsmt_num_cols = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
                 "BsmtFullBath", "BsmtHalfBath"]

for col in bsmt_num_cols:
    train_df[col] = train_df[col].fillna(0)
    test_df[col]  = test_df[col].fillna(0)

print("Basement numeric zeros filled.")


Basement numeric zeros filled.


## 10. Single-Missing Categoricals — Fill Mode

Test set has a few categoricals with 1–2 missing values. Fill with mode from train.

In [12]:
mode_cols = ["Electrical", "MSZoning", "KitchenQual", "Functional",
             "SaleType", "Utilities", "Exterior1st", "Exterior2nd"]

for col in mode_cols:
    mode_val = train_df[col].mode()[0]
    train_df[col] = train_df[col].fillna(mode_val)
    test_df[col]  = test_df[col].fillna(mode_val)

print("Mode-fill done for", len(mode_cols), "columns")


Mode-fill done for 8 columns


## 11. Remaining Numericals — Fill Median

⚠️ Use **train median** for both train and test (no data leakage).

In [13]:
numerical_cols = train_df.select_dtypes(include=["int64","float64"]).columns
# Exclude target
numerical_cols = [c for c in numerical_cols if c != "SalePrice"]

for col in numerical_cols:
    train_median = train_df[col].median()
    train_df[col] = train_df[col].fillna(train_median)
    test_df[col]  = test_df[col].fillna(train_median)   # ← train median for test too

print("Remaining numerical nulls filled with train median.")


Remaining numerical nulls filled with train median.


## 12. Verify — Zero Missing Values

In [14]:
train_nulls = train_df.isnull().sum().sum()
test_nulls  = test_df.isnull().sum().sum()

print(f"Train missing values: {train_nulls}")
print(f"Test missing values : {test_nulls}")

assert train_nulls == 0, "Train still has missing values!"
assert test_nulls  == 0, "Test still has missing values!"
print("✅ All missing values handled!")


Train missing values: 0
Test missing values : 0
✅ All missing values handled!


## 13. Save Clean Data

In [15]:
train_df.to_csv("../data/train_clean.csv", index=False)
test_df.to_csv("../data/test_clean.csv",  index=False)

print("✅ Data Cleaning Completed Successfully!")
print(f"   train_clean.csv → {train_df.shape}")
print(f"   test_clean.csv  → {test_df.shape}")


✅ Data Cleaning Completed Successfully!
   train_clean.csv → (1458, 80)
   test_clean.csv  → (1459, 79)
